In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

: 

In [ ]:
data = pd.read_csv("media/housing.csv")
data

In [ ]:
data.info()

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.info()

In [ ]:
from sklearn.model_selection import train_test_split

x =  data.drop(['median_house_value'], axis=1)
y = data['median_house_value']

In [ ]:
x

In [ ]:
y

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
print("y_test NaNs:", y_test.isnull().sum())

In [ ]:
train_data = x_train.join(y_train)
train_data

In [ ]:
train_data.hist(figsize=(15, 8))

In [ ]:
sns.heatmap(train_data.corr(numeric_only=True), annot=True, cmap="YlGnBu")

In [ ]:
train_data['total_rooms'] = np.log(train_data['total_rooms'] + 1)
train_data['total_bedrooms'] = np.log(train_data['total_bedrooms'] + 1)
train_data['population'] = np.log(train_data['population'] + 1)
train_data['households'] = np.log(train_data['households'] + 1)

In [ ]:
train_data.hist(figsize=(15,8))

In [ ]:
train_data = train_data.join(pd.get_dummies(train_data.ocean_proximity)).drop(['ocean_proximity'], axis=1)

# Convert all boolean columns to integers (0/1)
train_data = train_data.astype({col: int for col in train_data.columns if train_data[col].dtype == bool})

In [ ]:
train_data

In [ ]:
plt.figure(figsize=(15,8))
sns.heatmap(train_data.corr(numeric_only=True), annot=True, cmap="YlGnBu")

In [ ]:
plt.figure(figsize=(15,9))
sns.scatterplot(x="latitude", y="longitude", data=train_data, hue='median_house_value', palette='coolwarm')

Map of California, we can see that houses located on the coastline are more expensive.

In [ ]:
train_data['bedroom_ratio'] = train_data['total_bedrooms'] / train_data['total_rooms']
train_data['household_rooms'] = train_data['total_rooms'] / train_data['households']

In [ ]:
plt.figure(figsize=(15,8))
sns.heatmap(train_data.corr(numeric_only=True), annot=True, cmap="YlGnBu")

Using Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler() 

x_train, y_train = train_data.drop(['median_house_value'], axis=1), train_data['median_house_value']
x_train_s = scaler.fit_transform(x_train)

reg = LinearRegression()
reg.fit(x_train_s, y_train)

In [ ]:
test_data = x_test.join(y_test)

test_data['total_rooms'] = np.log(test_data['total_rooms'] + 1)
test_data['total_bedrooms'] = np.log(test_data['total_bedrooms'] + 1)
test_data['population'] = np.log(test_data['population'] + 1)
test_data['households'] = np.log(test_data['households'] + 1)

test_data = test_data.join(pd.get_dummies(test_data.ocean_proximity)).drop(['ocean_proximity'], axis=1)
test_data = test_data.astype({col: int for col in test_data.columns if test_data[col].dtype == bool})

test_data['bedroom_ratio'] = test_data['total_bedrooms'] / test_data['total_rooms']
test_data['household_rooms'] = test_data['total_rooms'] / test_data['households']


In [ ]:
x_test, y_test = test_data.drop(['median_house_value'], axis=1), test_data['median_house_value']

In [ ]:
x_test_s = scaler.transform(x_test)

In [ ]:
reg.score(x_test_s, y_test)

 This model of linear regression explains about 66.9% of the variance in the median house value on the test data.

Using Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

forest = RandomForestRegressor()

forest.fit(x_train_s, y_train)

In [ ]:
forest.score(x_test_s, y_test)

The Random Forest model explains about 82% of the variance in the house prices on your test set. In other words, the model can correctly "capture" or "predict" 82% of the reasons why house prices change. This is the R² score (coefficient of determination).

In [ ]:
from sklearn.model_selection import GridSearchCV

forest = RandomForestRegressor()

param_grid = {
    "n_estimators" : [3, 10, 30],
    "max_features" : [2, 4, 6, 8]
}

grid_search = GridSearchCV(forest, param_grid, cv=5, scoring="neg_mean_squared_error", return_train_score=True)

grid_search.fit(x_train_s, y_train)

In [ ]:
grid_search.best_estimator_

In [ ]:
forest = RandomForestRegressor(n_estimators=100, random_state=42)
forest.fit(x_train_s, y_train)

y_pred = forest.predict(x_test_s)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

In [ ]:
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")


In [ ]:
new_house = x_test.iloc[0] 
new_house_scaled = scaler.transform([new_house])


In [ ]:
predicted_price = forest.predict(new_house_scaled)[0]
print(f"Predicted price for the house: ${predicted_price:.2f}")